# **Thư viện**

In [2]:
import numpy as np
import pandas as pd
import underthesea
import re
import time
import unicodedata
import emoji
import string
from deep_translator import GoogleTranslator

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


# **Nạp dữ liệu**

In [3]:
pd.set_option("display.max_colwidth", None)
df = pd.read_csv("../data/raw/shopee_ratings.csv")
df.head(2)

,Id,Rating,Comment,ProductName,Helpfulness_Score,CommentDate,ProductUrl,SubCategory
0,97587241907,5,"Độ bền:dùng ổn định\nTốc độ đun:nahnh có màu đẹp\nThiết kế:xuất sắc, quá tuyệt vời, xứng đáng tỏng tầm fias","Ấm đun siêu tốc, Bình đun nước thủy tinh siêu tốc 2.5L, inox không gỉ dung tích 1.8L METIS tay cầm chống nóng",1,2026-05-05,https://shopee.vn/product/541665853/24422340309,Electronic
1,97586960285,5,Chất lượng sản phẩm:chất lượng khá ổn\n\nhộp hơi nhỏ nhưng vẫn ổn,"Hộp Đựng Cơm Túi Đựng Cơm Trưa Giữ Nhiệt Văn Phòng Cách Nhiệt, Combo 3 Ngăn Đựng Thức Ăn 2 Lớp Bằng Thép Không Gỉ 304",1,2026-05-05,https://shopee.vn/product/541665853/24025177675,Electronic


# **EDA**

In [ ]:
# Số lượng NaN
print("NaN:", df["Comment"].isna().sum())

# Chuỗi rỗng
empty_mask = (
    df["Comment"]
    .fillna("")
    .str.strip()
    .eq("")
)

print("Empty:", empty_mask.sum())

df.loc[empty_mask, ["Comment", "Rating"]].head(20)

NaN: 0
Empty: 1


,Comment,Rating
10173,,5


In [18]:
# Kiểm tra trùng lặp trên toàn bộ các cột của DataFrame
duplicate_mask = df.duplicated(keep=False)

print("Số bản ghi trùng lặp (tất cả các cột):", duplicate_mask.sum())
print("Số bản ghi độc lập (sau khi loại bỏ trùng hoàn toàn):", df.drop_duplicates().shape[0])

Số bản ghi trùng lặp (tất cả các cột): 0
Số bản ghi độc lập (sau khi loại bỏ trùng hoàn toàn): 20492


In [ ]:
# Tạo cột độ dài review

df["text_length"] = (
    df["Comment"]
    .fillna("")
    .str.len()
)

df["text_length"].describe()

count    20492.000000
mean       108.826225
std        128.317495
min          0.000000
25%         48.000000
50%         84.000000
75%        131.000000
max       6056.000000
Name: text_length, dtype: float64

In [ ]:
# Review quá ngắn

short_mask = df["Comment"].fillna("").str.strip().str.len() <= 5

print("Review <= 5 ký tự:", short_mask.sum())

Review <= 5 ký tự: 294


In [32]:
# Review chứa rating trực tiếp

rating_pattern = (
    r"\b(?:[1-5]\s*(?:sao|star|stars)"
    r"|(?:[1-9]|10)\s*/\s*10"
    r"|(?:[1-5]\s*/\s*5))\b"
)

df["has_explicit_rating"] = (
    df["Comment"]
    .fillna("")
    .str.contains(
        rating_pattern,
        case=False,
        regex=True
    )
)

# display(df[df["has_explicit_rating"] == True])
display(df["has_explicit_rating"].value_counts())

has_explicit_rating
False    20089
True       403
Name: count, dtype: int64

In [47]:
# Spam / quảng cáo

def spam_score(text: str) -> int:
    if not isinstance(text, str):
        return 0

    text = text.lower()

    score = 0

    # --------------------------------------------------------
    # 1. Dấu hiệu tin nhắn quảng cáo
    # --------------------------------------------------------
    if re.search(r"\[tb\]", text):
        score += 3

    # --------------------------------------------------------
    # 2. Cú pháp "soạn ... gửi ..."
    # --------------------------------------------------------
    if re.search(
        r"\bsoạn\s+\S+\s+gửi\s+\d{3,}",
        text
    ):
        score += 5

    # --------------------------------------------------------
    # 3. Số điện thoại / đầu số dịch vụ
    # --------------------------------------------------------
    if re.search(r"\bgọi\s+(?:19[0-9]|1800)\b", text):
        score += 4

    # --------------------------------------------------------
    # 4. Tin nhắn có rất nhiều chữ số
    # --------------------------------------------------------
    numbers = re.findall(r"\d+", text)

    if len(numbers) >= 4:
        score += 2

    # --------------------------------------------------------
    # 5. Dấu hiệu nhà mạng / gói cước
    # --------------------------------------------------------
    telecom_keywords = [
        "gb",
        "mb",
        "gói cước",
        "đăng ký",
        "gia hạn",
        "viettel",
        "vinaphone",
        "mobifone",
    ]

    telecom_count = sum(
        keyword in text
        for keyword in telecom_keywords
    )

    if telecom_count >= 2:
        score += 4

    # --------------------------------------------------------
    # 6. Ngôn ngữ quảng cáo
    # --------------------------------------------------------
    promotion_keywords = [
        "khuyến mãi",
        "ưu đãi",
        "tặng",
        "miễn phí",
        "chương trình",
    ]

    promotion_count = sum(
        keyword in text
        for keyword in promotion_keywords
    )

    if promotion_count >= 2:
        score += 2

    return score

df["spam_score"] = df["Comment"].apply(spam_score)

df["spam_score"].value_counts().sort_index()

df.loc[
    df["spam_score"] >= 5,
    ["Comment", "Rating", "spam_score"]
].sort_values(
    "spam_score",
    ascending=False
).head()

,Comment,Rating,spam_score
48,"[tb] mua 2 được 3, chơi lễ thả ga! soạn 5g10 gửi 191: 10k có 6gb sử dụng đến 24h ngày đăng ký, gia hạn theo ngày. đặc biệt: mua gói 2 lần trước 04/05 sẽ được tặng 1 lần đăng ký miễn phí. quà tặng áp dụng sau 1 ngày, khi có tin nhắn thông báo từ viettel. chương trình dành riêng cho quý khách. chi tiết gọi 198 (0đ).",5,20
17326,"[tb] bạn vừa hết data tốc độ cao? soạn edv gửi 9199 có ngay 1gb data lướt facebook, tiktok và 100% miễn phí combo khóa học kỹ năng mskill từ mobiedu! ưu đãi chỉ với 5.000đ/ngày. từ chối tư vấn cskh của mobifone, soạn tc 9199 gửi 9241.",4,16
8325,"[tb] tiktok thả ga, chỉ từ 5k! 1. soạn st7k gửi 191: 7k/ngày có 1gb 2. soạn t5k gửi 191: 5k/ngày thoải mái data truy cập tiktok (tối đa 15gb) ưu đãi sử dụng đến 24h ngày đăng ký. các gói cước gia hạn theo ngày. chi tiết lh 198 (0đ).",4,14
18887,[tb] không lo hết data! 1. soạn st7k gửi 191: 7k có 1gb sử dụng đến 24h ngày đăng ký. 2. soạn 5g20 gửi 191: 20k/3 ngày có 6gb. các gói cước gia hạn khi hết chu kỳ. chi tiết liên hệ 198 (0đ).,5,14
12584,"[tb] lên mạng thả ga, chỉ từ 7k! 1. soạn st7k gửi 191: 7k có 1gb. 2. soạn 5g10 gửi 191: 10k có 6gb. ưu đãi sử dụng đến 24h ngày đăng ký. các gói cước gia h",5,14


In [44]:
# Template "lấy xu"

reward_patterns = [
    r"ảnh.*chỉ mang tính chất.*xu",
    r"hình ảnh.*chỉ mang tính chất.*xu",
    r"video.*chỉ mang tính chất.*xu",
    r"mang tính chất nhận xu",
    r"mang tính chất lấy xu",
]

reward_regex = "|".join(reward_patterns)

df["is_reward_template"] = (
    df["Comment"]
    .fillna("")
    .str.contains(
        reward_regex,
        case=False,
        regex=True
    )
)

print(df["is_reward_template"].value_counts())

is_reward_template
False    20020
True       472
Name: count, dtype: int64


In [ ]:
# Phân bố độ dài review theo Rating
# Xem độ dài review tự nó đã có correlation với rating chưa

df.groupby("Rating")["text_length"].describe()

,count,mean,std,min,25%,50%,75%,max
Rating,,,,,,,,
1,5098.0,102.794821,111.876455,1.0,33.0,72.0,132.0,1273.0
2,3368.0,103.444181,99.479909,1.0,40.0,79.0,135.0,1421.0
3,3943.0,105.440528,122.780469,1.0,45.0,85.0,131.0,3781.0
4,3576.0,105.359899,107.815590,1.0,55.0,87.0,128.0,2654.0
5,4507.0,125.382738,174.803578,0.0,69.0,92.0,131.0,6056.0


In [48]:
# ============================================================
# EDA SUMMARY
# ============================================================

total = len(df)

eda_summary = pd.DataFrame({
    "Metric": [
        "Total reviews",
        "Missing",
        "Empty",
        "Duplicate",
        "Very short",
        "Explicit rating",
        "Spam candidates",
        "Reward template",
    ],

    "Count": [
        total,
        df["Comment"].isna().sum(),
        empty_mask.sum(),
        df["Comment"].duplicated().sum(),
        short_mask.sum(),
        df["has_explicit_rating"].sum(),
        (df["spam_score"] >= 5).sum(),
        df["is_reward_template"].sum(),
    ],
})

# Tính tỷ lệ %
eda_summary["Percentage (%)"] = (
    eda_summary["Count"] / total * 100
).round(2)

eda_summary

,Metric,Count,Percentage (%)
0,Total reviews,20492,100.00
1,Missing,0,0.00
2,Empty,1,0.00
3,Duplicate,1239,6.05
4,Very short,294,1.43
5,Explicit rating,403,1.97
6,Spam candidates,94,0.46
7,Reward template,472,2.30


# **Phân bố của Rating**

In [52]:
df_rating_summary = pd.DataFrame({
    "Count": df["Rating"].value_counts().sort_index(),
    "Percentage (%)": df["Rating"].value_counts(normalize=True).sort_index().mul(100).round(2)
})

# Thêm cột Cumulative Percentage bằng cách cộng dồn cột Percentage và làm tròn 2 chữ số
df_rating_summary["Cumulative (%)"] = df_rating_summary["Percentage (%)"].cumsum().round(2)

# Hiển thị bảng kết quả đẹp mắt
print(df_rating_summary)

        Count  Percentage (%)  Cumulative (%)
Rating                                       
1        5098           24.88           24.88
2        3368           16.44           41.32
3        3943           19.24           60.56
4        3576           17.45           78.01
5        4507           21.99          100.00


# **Xử lý emoji**

In [3]:
# ============================================================
# Hàm lấy tất cả emoji trong toàn bộ cột
# ============================================================
def extract_all_emojis(series):
  all_emojis = []
  for text in series.dropna():
    
    found = emoji.emoji_list(str(text))

    for item in found:
      all_emojis.append(item["emoji"])

  return all_emojis

df_emoji = pd.DataFrame(set(extract_all_emojis(df["Comment"])), columns=["raw_emoji"])
df_emoji.head()

,raw_emoji
0,💥
1,😙
2,📧
3,👊🏻
4,❤️‍🔥


In [4]:
# ============================================================
# Hàm chuyển đổi emoji thành mã hoá tiếng anh
# ============================================================
def convert_emoji_to_text(text):
  if pd.isna(text):
    return text
  # demojize chuyển emoji thành chữ, language='en' để lấy từ khóa tiếng Anh
  return emoji.demojize(str(text), language="en")

df_emoji["en_emoji"] = df_emoji["raw_emoji"].apply(convert_emoji_to_text)
df_emoji.head()

,raw_emoji,en_emoji
0,💥,:collision:
1,😙,:kissing_face_with_smiling_eyes:
2,📧,:e-mail:
3,👊🏻,:oncoming_fist_light_skin_tone:
4,❤️‍🔥,:heart_on_fire:


In [5]:
emoji_vi = {

    # -------------------------
    # Faces
    # -------------------------
    ":grinning_face:": "mặt cười",
    ":grinning_face_with_big_eyes:": "mặt cười mắt to",
    ":grinning_face_with_smiling_eyes:": "mặt cười mắt híp",
    ":beaming_face_with_smiling_eyes:": "mặt cười rạng rỡ",
    ":grinning_squinting_face:": "mặt cười nheo mắt",

    ":smiling_face:": "mặt cười",
    ":smiling_face_with_smiling_eyes:": "mặt cười với đôi mắt vui vẻ",
    ":smiling_face_with_hearts:": "mặt cười có trái tim",
    ":smiling_face_with_heart_eyes:": "mặt cười mắt hình trái tim",
    ":star_struck:": "mặt choáng ngợp",
    ":kissing_face:": "mặt hôn",
    ":kissing_face_with_smiling_eyes:": "mặt hôn với đôi mắt vui vẻ",
    ":kissing_face_with_closed_eyes:": "mặt hôn nhắm mắt",

    ":relieved_face:": "mặt nhẹ nhõm",
    ":happy_face:": "mặt vui vẻ",
    ":slightly_smiling_face:": "mặt hơi mỉm cười",
    ":upside_down_face:": "mặt lộn ngược",

    ":thinking_face:": "mặt suy nghĩ",
    ":confused_face:": "mặt bối rối",
    ":worried_face:": "mặt lo lắng",
    ":sad_face:": "mặt buồn",
    ":crying_face:": "mặt khóc",
    ":loudly_crying_face:": "mặt khóc lớn",
    ":angry_face:": "mặt tức giận",
    ":enraged_face:": "mặt giận dữ",
    ":fearful_face:": "mặt sợ hãi",
    ":screaming_face:": "mặt hét lên vì sợ",

    ":neutral_face:": "mặt bình thường",
    ":expressionless_face:": "mặt vô cảm",
    ":unamused_face:": "mặt không hài lòng",
    ":sleeping_face:": "mặt đang ngủ",
    ":sleepy_face:": "mặt buồn ngủ",
    ":dizzy_face:": "mặt chóng mặt",

    ":wink:": "mặt nháy mắt",
    ":winking_face:": "mặt nháy mắt",
    ":smirk:": "mặt cười nhếch mép",
    ":smiling_imp:": "mặt quỷ cười",
    ":imp:": "mặt quỷ",
    ":skull:": "đầu lâu",
    ":ghost:": "ma",
    ":alien:": "người ngoài hành tinh",
    ":robot:": "người máy",

    # -------------------------
    # Hearts
    # -------------------------
    ":red_heart:": "trái tim đỏ",
    ":orange_heart:": "trái tim cam",
    ":yellow_heart:": "trái tim vàng",
    ":green_heart:": "trái tim xanh lá",
    ":blue_heart:": "trái tim xanh dương",
    ":purple_heart:": "trái tim tím",
    ":black_heart:": "trái tim đen",
    ":white_heart:": "trái tim trắng",
    ":brown_heart:": "trái tim nâu",
    ":broken_heart:": "trái tim tan vỡ",
    ":two_hearts:": "hai trái tim",
    ":sparkling_heart:": "trái tim lấp lánh",
    ":growing_heart:": "trái tim lớn dần",
    ":beating_heart:": "trái tim đang đập",
    ":revolving_hearts:": "những trái tim xoay quanh nhau",
    ":heart_exclamation:": "trái tim cảm thán",
    ":heart_on_fire:": "trái tim rực lửa",
    ":mending_heart:": "trái tim đang được chữa lành",

    # -------------------------
    # Hands
    # -------------------------
    ":thumbs_up:": "ngón tay cái hướng lên",
    ":+1:": "ngón tay cái hướng lên",
    ":thumbs_down:": "ngón tay cái hướng xuống",
    ":-1:": "ngón tay cái hướng xuống",
    ":ok_hand:": "tay ra hiệu đồng ý",
    ":clapping_hands:": "hai tay vỗ tay",
    ":raised_hands:": "hai tay giơ lên",
    ":folded_hands:": "hai tay chắp lại",
    ":pray:": "hai tay cầu nguyện",
    ":victory_hand:": "bàn tay chữ V",
    ":crossed_fingers:": "bắt chéo ngón tay",
    ":handshake:": "bắt tay",
    ":waving_hand:": "vẫy tay",

    # -------------------------
    # People
    # -------------------------
    ":person:": "người",
    ":man:": "người đàn ông",
    ":woman:": "người phụ nữ",
    ":boy:": "cậu bé",
    ":girl:": "cô bé",
    ":baby:": "em bé",
    ":child:": "trẻ em",
    ":older_man:": "ông già",
    ":older_woman:": "bà già",

    # -------------------------
    # Animals
    # -------------------------
    ":dog:": "chó",
    ":cat:": "mèo",
    ":mouse:": "chuột",
    ":hamster:": "chuột hamster",
    ":rabbit:": "thỏ",
    ":fox:": "cáo",
    ":bear:": "gấu",
    ":panda:": "gấu trúc",
    ":koala:": "gấu koala",
    ":tiger:": "hổ",
    ":lion:": "sư tử",
    ":cow:": "bò",
    ":pig:": "lợn",
    ":frog:": "ếch",
    ":monkey:": "khỉ",
    ":chicken:": "gà",
    ":penguin:": "chim cánh cụt",
    ":bird:": "chim",
    ":fish:": "cá",
    ":whale:": "cá voi",
    ":dolphin:": "cá heo",

    # -------------------------
    # Food
    # -------------------------
    ":apple:": "quả táo",
    ":banana:": "quả chuối",
    ":watermelon:": "dưa hấu",
    ":grapes:": "nho",
    ":strawberry:": "dâu tây",
    ":peach:": "quả đào",
    ":cherries:": "quả anh đào",
    ":lemon:": "quả chanh",
    ":pizza:": "pizza",
    ":hamburger:": "hamburger",
    ":fries:": "khoai tây chiên",
    ":hotdog:": "bánh mì xúc xích",
    ":cake:": "bánh ngọt",
    ":birthday_cake:": "bánh sinh nhật",
    ":ice_cream:": "kem",
    ":coffee:": "cà phê",
    ":beer:": "bia",
    ":beers:": "hai cốc bia",

    # -------------------------
    # Objects
    # -------------------------
    ":phone:": "điện thoại",
    ":iphone:": "điện thoại",
    ":computer:": "máy tính",
    ":laptop:": "máy tính xách tay",
    ":keyboard:": "bàn phím",
    ":camera:": "máy ảnh",
    ":watch:": "đồng hồ",
    ":bulb:": "bóng đèn",
    ":book:": "quyển sách",
    ":books:": "những quyển sách",
    ":pencil:": "bút chì",
    ":pen:": "bút",
    ":key:": "chìa khóa",
    ":lock:": "ổ khóa",
    ":gift:": "quà tặng",

    # -------------------------
    # Nature
    # -------------------------
    ":sun:": "mặt trời",
    ":sunny:": "trời nắng",
    ":cloud:": "đám mây",
    ":rain_cloud:": "mây mưa",
    ":snowflake:": "bông tuyết",
    ":fire:": "lửa",
    ":water_wave:": "sóng nước",
    ":earth_globe:": "trái đất",
    ":star:": "ngôi sao",
    ":sparkles:": "lấp lánh",
    ":rainbow:": "cầu vồng",
    ":rose:": "hoa hồng",
    ":sunflower:": "hoa hướng dương",
    ":tulip:": "hoa tulip",
    ":four_leaf_clover:": "cỏ bốn lá",

    # -------------------------
    # Countries / flags
    # -------------------------
    ":England:": "cờ Anh",
    ":United_States:": "cờ Hoa Kỳ",
    ":United_Kingdom:": "cờ Vương quốc Anh",
    ":Vietnam:": "cờ Việt Nam",
    ":Japan:": "cờ Nhật Bản",
    ":South_Korea:": "cờ Hàn Quốc",
    ":China:": "cờ Trung Quốc",
    ":France:": "cờ Pháp",
    ":Germany:": "cờ Đức",
    ":Italy:": "cờ Ý",
    ":Spain:": "cờ Tây Ban Nha",
}

# ============================================================
# HÀM XỬ LÝ EN_EMOJI -> TỪ TIẾNG ANH
# ============================================================
def clean_emoji_name(value):

    if pd.isna(value):
        return ""

    value = str(value).strip()

    # Xóa dấu : ở đầu/cuối
    value = value.strip(":")

    # Thay _ bằng khoảng trắng
    value = value.replace("_", " ")

    # Xóa khoảng trắng thừa
    value = re.sub(r"\s+", " ", value)

    return value.strip()

# ============================================================
# HÀM DỊCH TỰ ĐỘNG
# ============================================================

# Nếu một emoji xuất hiện nhiều lần thì chỉ dịch 1 lần.
translation_cache = {}

translator = GoogleTranslator(
    source="en",
    target="vi"
)

def translate_emoji(value):

    if pd.isna(value):
        return ""

    original = str(value).strip()

    if original == "":
        return ""
    
    # Ưu tiên dictionary thủ công
    if original in emoji_vi:
        return emoji_vi[original]
    
    # Nếu đã dịch trước đó → lấy từ cache
    if original in translation_cache:
        return translation_cache[original]
    
    # Chuẩn hóa tên emoji
    english_name = clean_emoji_name(original)

    if english_name == "":
        return ""
    
    # Dịch tự động
    try:

        vietnamese = translator.translate(english_name)

        if vietnamese is None:
            vietnamese = english_name

        vietnamese = vietnamese.strip()

        # Cache kết quả
        translation_cache[original] = vietnamese

        # Nghỉ một chút để tránh gửi request quá nhanh
        time.sleep(0.1)

        return vietnamese

    except Exception as e:

        print(
            f"[WARNING] Không dịch được: {original} | "
            f"Lỗi: {e}"
        )

        translation_cache[original] = english_name

        return english_name

df_emoji["vi_emoji"] = df_emoji["en_emoji"].apply(translate_emoji)
df_emoji.to_csv('../data/dict/emoji_dict.csv', index=False, encoding='utf-8-sig')

df_emoji.head()

,raw_emoji,en_emoji,vi_emoji
0,💥,:collision:,va chạm
1,😙,:kissing_face_with_smiling_eyes:,mặt hôn với đôi mắt vui vẻ
2,📧,:e-mail:,e-mail
3,👊🏻,:oncoming_fist_light_skin_tone:,màu da sáng của nắm đấm sắp tới
4,❤️‍🔥,:heart_on_fire:,trái tim rực lửa


In [3]:
df_emoji_dict = pd.read_csv("../data/dict/emoji_dict.csv")
emoji_dict = df_emoji_dict.set_index('raw_emoji')['vi_emoji'].to_dict()

display(emoji_dict)

{'💥': 'va_chạm',
 '😙': 'hôn_trìu_mến',
 '📧': 'thư_điện_tử',
 '👊🏻': 'nắm_đấm',
 '❤️\u200d🔥': 'trái_tim_cháy_bỏng',
 '📛': 'thẻ_tên',
 '🗣️': 'lên_tiếng',
 '📌': 'đinh_ghim',
 '❤': 'trái_tim_yêu_thương',
 '🥹': 'nghẹn_ngào',
 '🌸': 'hoa_anh_đào',
 '💋': 'nụ_hôn',
 '💡': 'ý_tưởng',
 '💦': 'toát_mồ_hôi',
 '🥇': 'huy_chương_vàng',
 '🥞': 'bánh_kếp',
 '🤯': 'nổ_não',
 '🫶🏻': 'tạo_hình_trái_tim',
 '🤨': 'nhướn_mày',
 '☎️': 'điện_thoại_bàn',
 '🚀': 'tên_lửa',
 '👩\u200d❤️\u200d💋\u200d👨': 'nụ_hôn_đôi_lứa',
 '💩': 'bãi_phân',
 '🛒': 'giỏ_hàng',
 '😏': 'cười_khẩy',
 '🤌🏻': 'chụm_ngón_tay',
 '🥥': 'dừa',
 '✂️': 'kéo',
 '⛪': 'nhà_thờ',
 '🍯': 'mật_ong',
 '📥': 'hộp_thư_đến',
 '👏': 'vỗ_tay',
 '😆': 'cười_toe_toét',
 '😈': 'cười_ranh_mãnh',
 '💔': 'trái_tim_tan_vỡ',
 '🍦': 'kem_ốc_quế',
 '📈': 'biểu_đồ_tăng',
 '🤡': 'mặt_hề',
 '☎': 'điện_thoại_bàn',
 '🐕': 'chó',
 '👎🏼': 'không_thích',
 '🍀': 'cỏ_bốn_lá',
 '😮\u200d💨': 'thở_phào',
 '🤔': 'suy_nghĩ',
 '🤦\u200d♀️': 'bó_tay',
 '🥱': 'ngáp',
 '🔥': 'cực_hot',
 '‼️': 'chấm_than_kép',
 '👎🏻'

In [ ]:
def replace_emoji(text):
    emoji_pattern = "|".join(map(re.escape, emoji_dict.keys()))
    
    if pd.isna(text):
        return text

    return re.sub(
        emoji_pattern,
        lambda m: f" {emoji_dict[m.group()]} ", # Thêm khoảng trắng hai bên
        text
    )

df["Comment"] = df["Comment"].apply(replace_emoji)

df.to_csv('../data/processed/02_emoji_replated.csv', index=False, encoding='utf-8-sig')

# **Chuẩn hoá text**

In [4]:
# ============================================================
# HÀM CLEAN TEXT
# ============================================================
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    
    # --------------------------------------------------------
    # 1. Chuẩn hóa Unicode
    # --------------------------------------------------------
    text = unicodedata.normalize("NFC", text)

    # --------------------------------------------------------
    # 2. Chuyển về chữ thường
    # --------------------------------------------------------
    text = text.lower()

    # --------------------------------------------------------
    # 3. Xóa HTML
    # --------------------------------------------------------
    text = re.sub(r"<[^>]+>", " ", text)

    # --------------------------------------------------------
    # 4. Xóa URL
    # --------------------------------------------------------
    text = re.sub(
        r"https?://\S+|www\.\S+",
        " ",
        text
    )

    # --------------------------------------------------------
    # 5. Thay xuống dòng / tab bằng khoảng trắng
    # --------------------------------------------------------
    text = re.sub(r"[\r\n\t]+", " ", text)

    # --------------------------------------------------------
    # 6. Xoá "/" không nằm giữa các chữ số (ngày/tháng/năm)
    # --------------------------------------------------------
    text = re.sub(r"(?<!\d)/|/(?!\d)", " ", text)
    
    # --------------------------------------------------------
    # Chuẩn hóa ký tự lặp lại (đẹppppp -> đẹp, hayyyy -> hay)
    # --------------------------------------------------------
    text = re.sub(r'(.)\1{2,}', r'\1', text)

    # Xóa ký tự đặc biệt, giữ lại chữ, số, khoảng trắng và dấu !,?,/
    text = re.sub(r'[^\w\sÀ-ỹà-ỹ!?/]', ' ', text)

    # --------------------------------------------------------
    # Chuẩn hóa khoảng trắng
    # --------------------------------------------------------
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

df["Comment"] = df["Comment"].apply(clean_text)
df.to_csv('../data/processed/(2)_text_cleaned.csv', index=False, encoding='utf-8-sig')